# <span style="color:red; font-size:30px"> Lecture 18a: Merging </span>

<font size = "4">

- Datasets are often spread across mutiple files.

- Multi-file datasets can be visualized with an "Entity Relationship Diagram" (ERD). 

- An ERD depicts how the identifiers in each table are connected.

- Here is an example ERD for 3 files in the Formula One dataset

<img src="f1_data/erd_f1_simple.png" alt="drawing" width="600"/>

- Let's examine this relationship in more detail

In [ ]:
import pandas as pd

df_results = pd.read_csv("f1_data/results.csv")
df_races = pd.read_csv("f1_data/races.csv")
df_circuits = pd.read_csv("f1_data/circuits.csv")

In [ ]:
print("df_results:")
display(df_results)
print("df_races:")
display(df_races)
print("df_circuits:")
display(df_circuits)

<font size = "4">

- We can use these 3 separate data frames together to gain information about a race result.

In [ ]:
# pick one of the rows in results.csv between 0 and 25,659

result_row = 25
# result_row = 3000
# result_row = 24001
race_result = df_results.iloc[result_row, :]

print("race_result:", result_row)

# grab the raceId number
race_id = race_result["raceId"]
print("race ID:", race_id)

# use the raceId number to grab the corresponding row of the races.csv dataset
race_info = df_races.query("raceId == @race_id")

race_year = race_info["year"].item()
race_name = race_info["name"].item()
circuit_id = race_info["circuitId"].item()

print("Year:", race_year)
print("Name of Race:", race_name)

# use circuitId number to grab the corresponding row of the circuits.csv dataset
circuit_info = df_circuits.query("circuitId == @circuit_id")
circuit_name = circuit_info["name"].item()
circuit_loc = circuit_info["location"].item()
circuit_country = circuit_info["country"].item()

print("Circuit Name:", circuit_name)
print("Location:", circuit_loc)
print("Country:", circuit_country)

<font size = "4">

- Navigating between 3 different DataFrames is not ideal. 

- It would be nice if we could combine the DataFrames in some way.

- We can't "stack" them together (called concatenation) because the first DataFrame has 25,660 rows, the second has 1,079 rows, and the third only has 76 rows.

- However, we can "merge" them together using the Pandas function `pd.merge`

- We'll begin by working with some smaller DataFrames just for visualization's sake

In [ ]:
results_short = df_results[["resultId", "raceId", "driverId", "points"]]
races_short = df_races[["raceId", "year", "circuitId", "name"]]
circuits_short = df_circuits[["circuitId", "name", "location", "country"]]

print("results_short:")
display(results_short)
print("races_short:")
display(races_short)
print("circuits_short:")
display(circuits_short)

<font size = "4">

- We will combine `results_short` and `races_short` in the following way.

- Here are rows 0, 23, and 46 of `results_short` (you can get them using .iloc)


<div align="center">

|  | resultId | raceId | driverId | points |
| -------- | -------- | -------- | -------- | -------- |
| 0   | 1  | 18 | 1 | 10 |
| 23   | 24  | 19 | 9 | 8 |
| 46   | 47  | 20 | 9 | 6 |

</div>

- Here are rows 17, 18, and 19 of `races_short` (again, use .iloc)

<div align="center">

|  | raceId | year | circuitId | name |
| -------- | -------- | -------- | -------- | -------- |
| 17   | 18  | 2008 | 1 | Australian Grand Prix |
| 18   | 19  | 2008 | 2 | Malaysian Grand Prix |
| 19   | 20  | 2008 | 3 | Bahrain Grand Prix |

</div>

- After we use `pd.merge` to create a new DataFrame called `df_merged`, rows 0, 23, and 46 of the new DataFrame will be:

<div align="center">

|  | resultId | raceId | driverId | points | year | circuitId | name |
| -------- | -------- | -------- | -------- | -------- | -------- | -------- | -------- |
| 0   | 1  | 18 | 1 | 10 | 2008 | 1 | Australian Grand Prix | 
| 23   | 24  | 19 | 9 | 8 | 2008 | 2 | Malaysian Grand Prix |
| 46   | 47  | 20 | 9 | 6 | 2008 | 3 | Bahrain Grand Prix |

</div>


In [ ]:
df_merged = pd.merge(left = results_short, right = races_short, how = "left", on = "raceId")

display(df_merged)
print()
display(df_merged.iloc[[0, 23, 46], :])

<font size = "4">

- We used 4 keyword arguments in `pd.merge`

- "left" and "right" are the two DataFrames we are merging. The choice of which one is "left" and which one is "right" will affect the ultimate order of the columns in the merged DataFrame.

- "how" is a string that describes how the merge is performed. For this class, we will just focus on "left" and "right". (Other advanced choices include "outer", "inner", "cross", "left_anti", "right_anti")

- When we do `how = "left"`, we are basically treating the "left" DataFrame as the primary one. The order of the rows is chosen to maintain the order of the data in the left DataFrame.

- "on" is a string corresponding to the merging variable. In this case we are merging based on "raceId". Rows in the two datasets with identical values of raceId are matched, and the additional info from the "right" DataFrame is added on to the "left" DataFrame.

<font size = "4">

**Exercise:** Run the following cell and print/display the outputs to see how the merging procedure works for other choices of "left", "right", and "how".

In [ ]:
df_merged_2 = pd.merge(left = results_short, right = races_short, how = "right", on = "raceId")

df_merged_3 = pd.merge(left = races_short, right = results_short, how = "left", on = "raceId")

df_merged_4 = pd.merge(left = races_short, right = results_short, how = "right", on = "raceId")

<font size = "4">

- Next, we perform an additional merge so that we include the data from `circuits_short`

In [ ]:
df_merged_final = pd.merge(left = df_merged, right = circuits_short, how = "left", on = "circuitId")

display(df_merged_final.head())


<font size = "4">

<font size = "4">

Do you notice anything odd about the column names?

<font size = "4">

Instead of creating the additional variables `results_short`, `races_short`, and `circuits_short` we could have directly subsetted the desired columns when passing in the input arguments to `pd.merge`. 

In [ ]:
df_merged_direct = pd.merge(left = df_results[["resultId", "raceId", "driverId", "points"]],
                            right = df_races[["raceId", "year", "circuitId", "name"]],
                            how = "left",
                            on = "raceId")

df_merged_final_direct = pd.merge(left = df_merged_direct,
                                  right = df_circuits[["circuitId", "name", "location", "country"]],
                                  how = "left",
                                  on = "circuitId")

display(df_merged_final_direct.head())

# <span style="color:red"> Practice: Merging the 2020 Election Data </span>

<font size = "4">

- The folder `election_data` contains results from the 2020 U.S. general election (president, U.S. Senate, U.S. House, and governor), spread across 11 .csv files.

- Like the Formula One data, these files describe the same elections at different levels of detail. For the presidential race:

<div align="center">

| File | One row per... | Identifier(s) |
| -------- | -------- | -------- |
| president_state.csv | state | state |
| president_county.csv | county | state, county |
| president_county_candidate.csv | candidate in each county | state, county |

</div>

- The Senate and governor files follow the same pattern. The House files are organized by congressional district instead:

<div align="center">

| File | One row per... | Identifier(s) |
| -------- | -------- | -------- |
| house_state.csv | congressional district | district |
| house_candidate.csv | candidate in each district | district |

</div>

- In the county and district files, `current_votes` is the number of votes counted in that county or district. The candidates' vote totals add up to `current_votes`.

- Run the cell below to read in the files we'll use.

In [ ]:
df_pres_state = pd.read_csv("election_data/president_state.csv")
df_pres_county = pd.read_csv("election_data/president_county.csv")
df_pres_cand = pd.read_csv("election_data/president_county_candidate.csv")

df_gov_state = pd.read_csv("election_data/governors_state.csv")
df_senate_state = pd.read_csv("election_data/senate_state.csv")

df_house_state = pd.read_csv("election_data/house_state.csv")
df_house_cand = pd.read_csv("election_data/house_candidate.csv")

<font size = "4">

**Exercise 1:** Navigating between DataFrames (like we did at the start of lecture)

- Display `df_house_cand` and `df_house_state`. Print the column names of both DataFrames, and print the number of rows of each with .shape.

- Pick one row of `df_house_cand` (any row between 0 and 1,214). Use its `district` value and `.query()` to grab the corresponding row of `df_house_state`.

- Print the candidate's name, party, and number of votes, along with the number of votes counted in their district (`current_votes`). What percent of the district's votes did the candidate receive?

In [ ]:
# your answer here

<font size = "4">

**Exercise 2:** Merging the House data

- Use `pd.merge` to add the `current_votes` column from `df_house_state` onto `df_house_cand`. Keep every row of `df_house_cand` and call the result `df_house_merged`.

- Only bring in the columns `district` and `current_votes` from `df_house_state`, by subsetting the columns directly inside `pd.merge`.

- Create a new column `vote_pct` equal to each candidate's percent of the votes in their district.

- Use `.query()` to display the rows for `"Delaware at large"`. Print the sum of vote_pct for those rows.

- Print the `.shape` of df_house_cand and of df_house_merged.

In [ ]:
# your answer here

<font size = "4">

**Exercise 3:** Matching column names

- Merge `df_pres_county` (left) with `df_pres_state` (right) using `how = "left"` and `on = "state"`. Call the result `df_pres_merged`.

- Print the column names. Both DataFrames had a column called `total_votes`. Print the column names of df_pres_merged.

- Create a new column called `pct_of_state` with the percent of the state's total votes that came from each county.

- Use `.query()` to find all counties that cast more than 25% of their state's votes.

- `df_pres_state` has a row with `state` equal to `"United States"`. Use `.query()` to display any rows of df_pres_merged where state is "United States".

In [ ]:
# your answer here

<font size = "4">

**Exercise 4:** "left" vs. "right"

Only 11 states held an election for governor in 2020.

- Merge `df_pres_state` and `df_gov_state` on `"state"` twice: once with `how = "left"` and once with `how = "right"`. Keep `df_pres_state` as the left DataFrame both times.

- How many rows does each result have?

- In the `how = "left"` result, what value appears in the `votes` column for a state like Georgia that did not elect a governor in 2020?

- Create a DataFrame df_gov_only that contains only the 11 states with a governor’s race, using pd.merge, and print its shape.

In [ ]:
# your answer here

<font size = "4">

**Exercise 5:** When a merge adds rows

- `df_pres_state` has 52 rows. Merge it (as the left DataFrame) with `df_senate_state` using `how = "left"` and `on = "state"`.

- Print the .shape of the result. Then use `.query()` to display the rows of df_senate_state and of the merged result where state is "Georgia".

In [ ]:
# your answer here

<font size = "4">

**Exercise 6 (Challenge):** Merging on more than one column

- We want to add each county's `current_votes` from `df_pres_county` onto `df_pres_cand`, so we can compute each candidate's percent of the vote in each county.

- First try merging on `"county"` only (use `how = "left"`, with `df_pres_cand` on the left, and only bring in the `county` and `current_votes` columns from `df_pres_county`). How many rows does the result have compared with `df_pres_cand`?

- Use `.query()` on `df_pres_county` to look up `"Washington County"`. Use .query() to display the rows of df_pres_county where county is "Washington County", and print how many there are.

- The `on` argument can also be a *list* of column names. Rows are matched only if they agree on **every** column in the list. Redo the merge with `on = ["state", "county"]` (and bring in the `state` column too). Check the number of rows.

- Create a column `vote_pct` with each candidate's percent of the vote in their county, then display the rows for Fulton County, Georgia.

In [ ]:
# your answer here

In [ ]:
# your answer here